# Análisis Inferencial


* Luisa Merlo García | A01067715
* Edgar Osvaldo Navarro | A01644488
* Ariana Guadalupe Rosales Villalobos | A01644773


## Ejercicio 1 (60 puntos)

La base de datos Palmer PenguinsDescargar Palmer Penguins contiene información sobre 344 pingüinos observados en el archipiélago Palmer, en la Antártida. Incluye ejemplares de tres especies distintas y registra tanto características físicas como información sobre el lugar y el año de observación, por lo que permite comparar grupos y estudiar relaciones entre variables.

Para esta base de datos, analiza lo siguiente:

### A. Masa corporal

Analiza la variable masa corporal (body_mass_g) y realiza lo siguiente:

* Evalúa el supuesto de normalidad de la masa corporal para cada especie mediante una prueba de normalidad apropiada. Complementa el resultado con una inspección gráfica, por ejemplo mediante histogramas o gráficos Q–Q.
* Determina, mediante una prueba de hipótesis, si la masa corporal media de los pingüinos Adelie es diferente de 3800 g.
* Determina si existen diferencias significativas en la masa corporal media entre las tres especies.
* Considerando únicamente a los pingüinos Adelie, determina si existen diferencias significativas en la masa corporal media entre las islas Biscoe, Dream y Torgersen.
* Analiza la masa corporal considerando simultáneamente los factores especie y sexo. Determina si existe un efecto significativo de la especie, un efecto significativo del sexo y una interacción significativa entre ambos factores.


In [13]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

penguins = pd.read_csv("penguins.txt")

penguins.groupby("species")["body_mass_g"].count()

species
Adelie       151
Chinstrap     68
Gentoo       123
Name: body_mass_g, dtype: int64

In [14]:
# Construye un intervalo de confianza del 95 % para la  masa corporal media de todos los pingüinos.
body_mass = penguins["body_mass_g"].dropna()

x = body_mass
cl = 0.95
alpha = 1 - cl

# Calcular std y media poblacional
sample_mean = np.mean(x)
sample_std_dev = np.std(x, ddof=1)
sample_size = len(x)

# Calcular el valor critico de distribucion t
t_score = stats.t.ppf(1 - alpha/2, df=sample_size - 1)

# Calcular el margen de error y el intervalo de confianza
margin_of_error = t_score * sample_std_dev / np.sqrt(sample_size)

confidence_interval = (
    sample_mean - margin_of_error,
    sample_mean + margin_of_error
)

print("Sample size =", sample_size)
print("Sample mean =", sample_mean)
print("Sample standard deviation =", sample_std_dev)
print("t-score =", t_score)
print("Margin of error =", margin_of_error)
print("95% Confidence Interval =", confidence_interval)

Sample size = 342
Sample mean = 4201.754385964912
Sample standard deviation = 801.9545356980955
t-score = 1.9669451293272178
Margin of error = 85.29605394086734
95% Confidence Interval = (np.float64(4116.4583320240445), np.float64(4287.05043990578))


In [15]:
## Construye intervalos de confianza del 95 % para la masa corporal media de cada especie.
species_list = penguins["species"].unique()

cl = 0.95
alpha = 1 - cl

for species in species_list:
    
    # Seleccionar los valores de masa corporal de las especies
    x = penguins.loc[
        penguins["species"] == species,
        "body_mass_g"
    ].dropna()

    # Estadísticas de la muestra
    sample_mean = np.mean(x)
    sample_std_dev = np.std(x, ddof=1)
    sample_size = len(x)

    # Calcular el valor crítico de la distribución t
    t_score = stats.t.ppf(
        1 - alpha/2,
        df=sample_size - 1
    )

    # Margen de error
    margin_of_error = (
        t_score
        * sample_std_dev
        / np.sqrt(sample_size)
    )

    # Intervalo de confianza
    confidence_interval = (
        sample_mean - margin_of_error,
        sample_mean + margin_of_error
    )

    print(species)
    print("Sample size =", sample_size)
    print("Sample mean =", sample_mean)
    print("95% Confidence Interval =", confidence_interval)
    print()

Adelie
Sample size = 151
Sample mean = 3700.662251655629
95% Confidence Interval = (np.float64(3626.926242277889), np.float64(3774.398261033369))

Gentoo
Sample size = 123
Sample mean = 5076.016260162602
95% Confidence Interval = (np.float64(4986.034279556633), np.float64(5165.99824076857))

Chinstrap
Sample size = 68
Sample mean = 3733.0882352941176
95% Confidence Interval = (np.float64(3640.0593266448386), np.float64(3826.1171439433965))



In [16]:
## Construye un intervalo de confianza del 95 % para la diferencia de masa corporal media entre machos y hembras.


male = penguins.loc[
    penguins["sex"] == "male",
    "body_mass_g"
].dropna()

female = penguins.loc[
    penguins["sex"] == "female",
    "body_mass_g"
].dropna()

cl = 0.95
alpha = 1 - cl

# Calcular la media y desviación estándar de cada grupo
mean_male = np.mean(male)
std_male = np.std(male, ddof=1)

mean_female = np.mean(female)
std_female = np.std(female, ddof=1)

n_male = len(male)
n_female = len(female)

# Calcular el error estándar de la diferencia de medias
standard_error_diff = np.sqrt(
    std_male**2 / n_male +
    std_female**2 / n_female
)

# gardos d confiaza Welch-Satterwhaite
df = (
    std_male**2 / n_male +
    std_female**2 / n_female
)**2 / (
    (std_male**2 / n_male)**2 / (n_male - 1)
    +
    (std_female**2 / n_female)**2 / (n_female - 1)
)

# valor critico t 
t_score = stats.t.ppf(1 - alpha/2, df=df)

# Magen de error
margin_of_error_diff = t_score * standard_error_diff

# Intervalo de confianza para macho y hembra
confidence_interval_diff = (
    mean_male - mean_female - margin_of_error_diff,
    mean_male - mean_female + margin_of_error_diff
)

print("Male sample size =", n_male)
print("Female sample size =", n_female)

print("Male mean =", mean_male)
print("Female mean =", mean_female)

print("Difference male - female =", mean_male - mean_female)

print("Degrees of freedom =", df)

print(
    "95% Confidence Interval for male - female =",
    confidence_interval_diff
)

Male sample size = 168
Female sample size = 165
Male mean = 4545.684523809524
Female mean = 3862.2727272727275
Difference male - female = 683.4117965367964
Degrees of freedom = 323.8958810286484
95% Confidence Interval for male - female = (np.float64(526.2453315709475), np.float64(840.5782615026452))


### B. Longitud de la aleta

Analiza la variable longitud de la aleta (flipper_length_mm):

* Estima mediante intervalos de confianza del 95 % la longitud media de la aleta para cada especie.
* Construye un intervalo de confianza para la diferencia en la longitud media de la aleta entre Gentoo y Adelie.
* Evalúa si la longitud media de la aleta de los pingüinos Gentoo es mayor que 210 mm.
* Evalúa el supuesto de normalidad de la longitud de la aleta para cada especie mediante una prueba de normalidad apropiada. Complementa el resultado con una inspección gráfica, por ejemplo mediante histogramas o gráficos Q–Q.
* Determina si existen diferencias significativas en la longitud media de la aleta entre las tres especies.
* Considerando únicamente a los pingüinos Adelie, determina si existen diferencias significativas en la masa corporal media entre las islas Biscoe, Dream y Torgersen.
* Analiza la longitud media de la aleta considerando simultáneamente los factores especie y sexo. Determina si existe un efecto significativo de la especie, un efecto significativo del sexo y una interacción significativa entre ambos factores.


### C. Longitud y profundidad del pico

Utilizando las variables bill_length_mm y bill_depth_mm:

* Construye intervalos de confianza del 95 % para la longitud media y la profundidad media del pico.
* Construye intervalos de confianza para la diferencia de medias entre machos y hembras en ambas características.
* Evalúa el supuesto de normalidad de la longitud media del pico para cada especie mediante una prueba de normalidad apropiada. Complementa el resultado con una inspección gráfica, por ejemplo mediante histogramas o gráficos Q–Q.
* Evalúa el supuesto de normalidad de la profundidad media del pico para cada especie mediante una prueba de normalidad apropiada. Complementa el resultado con una inspección gráfica, por ejemplo mediante histogramas o gráficos Q–Q.
* Determina si la longitud media del pico difiere entre las especies Adelie y Chinstrap.
* Determina si existen diferencias significativas en la profundidad media del pico entre las tres especies.
* Considerando únicamente a los pingüinos Adelie, determina si existen diferencias significativas en la produndidad media del pico entre las islas Biscoe, Dream y Torgersen.
* Analiza la longitud media del pico considerando simultáneamente los factores especie y sexo. Determina si existe un efecto significativo de la especie, un efecto significativo del sexo y una interacción significativa entre ambos factores.



### D. Proporciones

Utilizando las variables categóricas del conjunto de datos:

* Construye un intervalo de confianza del 95 % para la proporción de pingüinos Gentoo.
* Construye un intervalo de confianza del 95 % para la proporción de machos.
* Construye un intervalo de confianza para la diferencia entre la proporción de machos en las especies Adelie y Gentoo.
* Determina si existe evidencia suficiente para concluir que más del 30 % de los pingüinos pertenecen a la especie Gentoo.
* Evalúa si la proporción de machos es la misma en Adelie y Gentoo.


### E. Relaciones entre variables

Analiza la relación entre las características físicas de los pingüinos:

* Estima la correlación entre masa corporal y longitud de la aleta.
* Prueba si existe una correlación lineal significativa entre ambas variables.
* Realiza un análisis similar entre longitud del pico y masa corporal.
* Determina si existe asociación entre la especie y la isla donde fue observado el pingüino.
* Determina si existe asociación entre especie y sexo.


## Ejercicio 2

## Ejercicio 3